# 21. Roundtrip route distances

**Medium**

For every route with a reverse route, return one roundtrip row with from lexically smaller than to and roundtrip_dist equal to the sum of both directional distances. Directional routes are unique; ignore one-way routes.


In [0]:
%run ./_setup $exercise_id="21"


**Your solution:** add imports and return your result from `solve`.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def solve(input_df):
    # Write your solution here.
    result_df = (
        input_df.alias("a")
        .join(
            input_df.alias("b"),
            ((F.col("a.from") == F.col("b.to")) & (F.col("a.to") == F.col("b.from"))),
            how="inner",
        )
        .withColumn("roundtrip_dist", F.col("a.dist") + F.col("b.dist"))
        .withColumn("trip", 
                    F.array_sort(F.array(F.col("a.from"), F.col("a.to")))
                    )
        .withColumn("rank", F.row_number().over(Window.partitionBy(F.col("trip")).orderBy("a.from")))
        .filter(F.col("rank")==1)
        .select("a.from","a.to","roundtrip_dist")
    )
    return result_df


solve(input_df).display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def solve(input_df):
    return (
        input_df.alias("a")
        .join(input_df.alias("b"),
            (
                (F.col("a.from")==F.col("b.to")) 
                &
                (F.col("a.to")==F.col("b.from"))
                &
                (F.col("a.from")<F.col("a.to"))
            ),
            how="inner"
        )
        .select("a.from","a.to",(F.col("a.dist")+F.col("b.dist")).alias("roundtrip_dist"))
    )

solve(input_df).display()

**Check:** run below to test all cases and save progress.


In [0]:
check_report = check_solution(exercise_id, solve, spark)


In [0]:
show_progress(exercise_id, spark)
